# Course 2 — Logistic Regression I

From log-odds to ROC curves on the `Default` dataset, with a Smarket
detour to see what 'no signal' looks like.

**Sections**
1. From linear regression to log-odds (0:00–0:30)
2. Multiple logistic regression on Smarket (0:30–1:00)
3. predict_proba, ROC, and threshold tuning (1:00–1:30)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, roc_curve, auc,
                              confusion_matrix, f1_score)
default = load_dataset('default')
smarket = load_dataset('smarket')
d = default.copy(); d['y'] = (d['default'] == 'Yes').astype(int)
print('default:', d.shape, '  smarket:', smarket.shape)


## 1. From linear regression to log-odds

$$p(x) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x)}}, \qquad \log\frac{p}{1-p} = \beta_0 + \beta_1 x$$

Linear in log-odds, S-shaped in probability. The log-likelihood
$\ell(\beta) = \sum_i [y_i \log p_i + (1-y_i) \log(1-p_i)]$ is
concave — a fast maximizer always finds the unique optimum.

In [ ]:
z = np.linspace(-6, 6, 200)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(z, 1/(1+np.exp(-z)))
ax.set_xlabel(r'$\beta_0 + \beta_1 x$ (log-odds)')
ax.set_ylabel('p(x)'); ax.set_title('The sigmoid'); plt.show()


### Fit on `default ~ balance`

In [ ]:
X = d[['balance']].to_numpy(); y = d['y'].to_numpy()
m = LogisticRegression(C=1e6, max_iter=2000).fit(X, y)  # huge C -> ~unregularized
print(f'intercept = {m.intercept_[0]:.4f}')
print(f'slope     = {m.coef_[0][0]:.6f}')
print(f'odds ratio per +1 unit of balance: {np.exp(m.coef_[0][0]):.6f}')
print(f'odds ratio per +100 units:          {np.exp(100*m.coef_[0][0]):.4f}')


In [ ]:
xs = np.linspace(0, 2700, 200).reshape(-1, 1)
ps = m.predict_proba(xs)[:, 1]
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X.ravel(), y, s=4, alpha=0.2)
ax.plot(xs, ps, color='C3', linewidth=2)
ax.set_xlabel('balance'); ax.set_ylabel('P(default)')
ax.set_title('Logistic fit'); plt.show()


## 2. Multiple LR on Smarket

Predict `Direction` (Up / Down) from the lag returns *only* —
`Today` is the answer, leaking it would be cheating.

In [ ]:
Xs = smarket[['Lag1', 'Lag2', 'Lag3', 'Lag4', 'Lag5', 'Volume']]
ys = (smarket['Direction'] == 'Up').astype(int)
m = LogisticRegression(C=1e6, max_iter=2000).fit(Xs, ys)
print(pd.Series(m.coef_[0], index=Xs.columns).round(4).to_string())
print(f'training accuracy = {m.score(Xs, ys):.4f}')


Coefficients are tiny, accuracy barely above 50%. *Markets are hard.*
Use a stratified train/test split to confirm it's not just noise that happens to overfit the training set.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.3, random_state=0, stratify=ys)
m = LogisticRegression(C=1e6, max_iter=2000).fit(Xtr, ytr)
print(f'held-out accuracy = {m.score(Xte, yte):.4f}')


## 3. predict_proba, ROC, and threshold tuning

In [ ]:
X = d[['balance', 'income']].to_numpy(); y = d['y'].to_numpy()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
m = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
proba = m.predict_proba(Xte)[:, 1]
print('first 5 probs :', proba[:5].round(3))
print('first 5 preds :', (proba >= 0.5).astype(int)[:5])


### ROC curve

In [ ]:
fpr, tpr, thr = roc_curve(yte, proba)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr); ax.plot([0,1], [0,1], 'k--', alpha=0.4)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title(f'ROC — AUC = {auc(fpr, tpr):.4f}'); plt.show()


### Sweep thresholds, pick the F1-max

In [ ]:
ts = np.linspace(0.02, 0.6, 50)
f1s = [f1_score(yte, (proba >= t).astype(int)) for t in ts]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ts, f1s); best_t = ts[int(np.argmax(f1s))]
ax.axvline(best_t, color='C3', linestyle='--', label=f'best t = {best_t:.2f}')
ax.set_xlabel('threshold'); ax.set_ylabel('F1'); ax.legend()
ax.set_title('Threshold tuning'); plt.show()
print(f'F1 at default t=0.5: {f1_score(yte, (proba>=0.5).astype(int)):.4f}')
print(f'F1 at best t = {best_t:.2f}: {max(f1s):.4f}')


### Recap
- Logistic regression = linear model in log-odds, sigmoid in probability.
- e^β is the odds-ratio per unit change of the predictor.
- `predict_proba` gives the score; `predict` applies a 0.5 cutoff.
- Tune the cutoff with F1 (or any business cost ratio) — 0.5 is rarely
  the right answer on imbalanced data.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

**Task 1.** Load `titanic`, drop rows missing `age`, one-hot encode
`sex` and `embarked`. Fit binary logistic regression predicting
`survived` from `pclass, sex, age, sibsp, parch, fare, embarked`.
Report held-out accuracy and AUC.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
# your code here


### Exercise 1 — Solution

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
df = load_dataset('titanic').dropna(subset=['age']).reset_index(drop=True)
df = pd.get_dummies(df[['survived','pclass','sex','age','sibsp','parch','fare','embarked']].fillna(df.median(numeric_only=True)),
                     columns=['sex','embarked'], drop_first=True, dtype=float)
y = df['survived']; X = df.drop(columns=['survived'])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
m = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
print(f'accuracy = {m.score(Xte, yte):.4f}')
print(f'AUC      = {roc_auc_score(yte, m.predict_proba(Xte)[:,1]):.4f}')


---

## Exercise 2

**Task 2.** Tune the decision threshold for maximum F1 on the held-out set.

In [ ]:
# your code here


### Exercise 2 — Solution

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
p = m.predict_proba(Xte)[:, 1]
ts = np.linspace(0.1, 0.9, 81)
f1s = [f1_score(yte, (p >= t).astype(int)) for t in ts]
best_t = ts[int(np.argmax(f1s))]
print(f'F1@0.5     = {f1_score(yte, (p>=0.5).astype(int)):.4f}')
print(f'F1@{best_t:.2f}  = {max(f1s):.4f}')


---

## Exercise 3

**Task 3.** Add an interaction `sex_male × pclass` to the design.
Does AUC improve?

In [ ]:
# your code here


### Exercise 3 — Solution

In [ ]:
X2 = X.copy()
X2['sex_x_pclass'] = X['sex_male'] * X['pclass']
Xtr2, Xte2, ytr2, yte2 = train_test_split(X2, y, test_size=0.3, random_state=0, stratify=y)
m2 = LogisticRegression(max_iter=2000).fit(Xtr2, ytr2)
print(f'baseline AUC      = {roc_auc_score(yte, m.predict_proba(Xte)[:,1]):.4f}')
print(f'with interaction  = {roc_auc_score(yte2, m2.predict_proba(Xte2)[:,1]):.4f}')
